In [ ]:
!pip install langchain langchain_community langchain_huggingface faiss-cpu

In [8]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_community.vectorstores import FAISS

In [9]:
# Step 1: load raw PDF(s)
DATA_PATH="data/"
def load_pdf_files(data):
    loader = DirectoryLoader(data,
                             glob='*.pdf',
                             loader_cls=PyPDFLoader)
    
    documents=loader.load()
    return documents

In [ ]:
documents=load_pdf_files(data=DATA_PATH)
# print("Length of PDF pages: ", len(documents))

In [12]:
# Step 2: Create Chunks 
def create_chunks(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,
                                                 chunk_overlap=50)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [13]:
text_chunks=create_chunks(extracted_data=documents)
print("Length of Chunks:", len(text_chunks))

Length of Chunks: 7079


In [14]:
# Step 3: Create Vector Embeddings 
def get_embedding_model(): 
    embeddings_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings_model 

In [15]:
embedding_model=get_embedding_model()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
# Step 4: Store embeddings in FAISS  
DB_FAISS_PATH="vectorstore/db_faiss"
db=FAISS.from_documents(text_chunks, embedding_model)
db.save_local(DB_FAISS_PATH)